# Shared-weight pair readout — I · T4×2 · endpoint500
**Internet ON · T4×2 · Save & Run All. Một tài khoản chạy I, tài khoản còn lại chạy J.**

Hai nhánh dùng cùng attention cặp, đặc trưng, tham số, loss và khởi tạo.
I đọc thực thể bằng priors riêng; J dùng marginals của attention cặp.
Đây là cải tiến tích hợp kế thừa, chưa có kết quả hoặc chứng nhận tính mới.

Add Input sáu dataset: `relation-frames-v2-cache-v105`, `pair-source-fullframe-ce`,
`mscoco-2014` của vuthetam, `prompt-cache`, `visual-cache`, `objectdetectionecache`.
Không cần UPT, checkpoint frame200, API token hoặc ảnhRGB. Chỉ COCO JSON được đọc từ mscoco.
Đường dẫn điền sẵn ở cell tiếp theo; sửa mount nếu khác, giữ nguồn/SHA.

Pipeline: kiểm phụ thuộc metric → sao chép cache612+dev1000 trên CPU → actual-model2GPU preflight
→ baseline A hiện tại →500updates → caption A/end/off và heldout → metrics/ZIP.
CPU prepare cap30phút, pha2GPU cap60phút/nhánh, CPU metrics cap15phút; đều là giới hạn, không ETA.
Dừng khi lỗi/cap, không tự kéo dài hoặc đổi cấu hình. Giữ checkpoint endpoint.bin nếu đã có.
User báo còn25giờ, không suy thành50giờ hoặc khẳng định quota riêng của tài khoản khác.


In [ ]:
from pathlib import Path
ARM='I'
CACHE=Path('/kaggle/input/datasets/ducanh2403/relation-frames-v2-cache-v105')
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT=Path('/kaggle/working/shared_pair_readout_'+ARM)
REPORTED_REMAINING_HOURS=25  # User report; account distribution and Kaggle accounting not inferred.


In [ ]:
import hashlib,json,os,signal,subprocess,sys,tempfile,threading,time,traceback,zipfile
os.environ.update(CUBLAS_WORKSPACE_CONFIG=':4096:8',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',
    PYTHONUNBUFFERED='1',PYTHONHASHSEED='42',TOKENIZERS_PARALLELISM='false',HDF5_USE_FILE_LOCKING='FALSE')
SOURCE_COMMIT='a00557e431a58c1171cd81b13b2d11f477ae499e'
SOURCE_ASSETS={'experiments/shared_pair_readout/readout_data.py': '3bdc5e38c942cfeb13f8a83e409e367d6cba1b2fb5da6149c2d400b0c51b889c', 'experiments/shared_pair_readout/prepare_readout.py': '96cc8200ec0b7c85a1544cac202dc0aef35d6fe2d45cbc94ac75da3be5187991', 'experiments/shared_pair_readout/readout_model.py': '69ad8215836852e560e42ed741d8fc01350d920b97e2e2ce4a72da47d17a2031', 'experiments/shared_pair_readout/run_readout.py': '711f11a2bf83c0783f5af0df88ecf53a57e6bbb904472904187ca7a02363204b', 'experiments/shared_pair_readout/evaluate_readout.py': '6cfdcaba654aaacc736cc4eb208e82da1256d6d990432e63e9663fe6be76d74c', 'experiments/shared_pair_readout/metric_preflight.py': '6e29f54e57b0b2e7f6f62de57017061071444d254b09305411716713b86f83f4', 'experiments/shared_pair_readout/test_readout_data.py': '9d5f5adde091b16350b4e2131d7846244e85328562ee350bdcde8ef9aad506f7', 'experiments/shared_pair_readout/DEV_DIMENSIONS.json': '534c0501884863464bb10c3bb2f504bd69ba9a290c4345ebf3ee179ad3c60206', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/models.py': '18026877836052c83e75c0d896adcc89822bb35616173c7e8b14b17c9a22e9d8', 'captioning/config.py': '4766d1bf82bf82b5199616c7d90ca74242f011e26f08db21b154b1c7d58e72bc', 'captioning/grounded_relations.py': '2c6b15048dce17a9bfa596f02af78866f0b7978f65d172414a3648c9aa49c58f', 'captioning/decoder_relation_memory.py': 'bc5086915bf8f35cf7161311785cf612b25695c5f0dbe03a760d8aca724dfc82', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'captioning/tokenizer.py': 'df2ace0025aa7d4e12c289ab7bca68046b48d4b6c2b5c12606a19ff5a50a638e', 'captioning/object_reward.py': '23282b1faca0b8e177ad8ce300f0b78ad4a5ab7b100e4b359b044025c567800c', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/visual_cache.py': 'adbcc9678fff499fc767cdbf04a4966186a40729f51c5bfb2525e79f9b6b5fe1', 'captioning/semantic_prompts.py': 'a06f12dea63604f9db9d19b6922815db27a35b648a17059407ffaf26a2a0d511', 'experiments/relation_frame_pretrain/frame_data.py': '87c58a50142f68b1721323f7c53fa08e256bb5bc8aee3d7625262161ae1692de', 'experiments/relation_frames_inventory/V2_MANIFEST.json.gz': '6ca6af23d0a31043fecdf33398f5b60098be07cbeea9fb91cbfec4b314c2cfa7', 'experiments/semantic_grounding_method/data/gain_metadata.json.gz': '62f43f771aec9feba31a2fbbb6fe7db2878f259979faac378b74a754822c7b67'}
REPO=Path(tempfile.mkdtemp(prefix='shared-pair-'+ARM+'-'))/'repo'
LOG=OUTPUT.with_name(OUTPUT.name+'_notebook.log')
REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
if OUTPUT.exists():raise FileExistsError('Giữ output cũ; dùng phiên mới trước khi chạy lại.')
if LOG.exists() or REVIEW.exists():raise FileExistsError('Giữ log/review cũ; đổi OUTPUT hoặc dùng phiên mới.')
for p in (CHECKPOINT,COCO_JSON,LEGACY_PROMPT,DETECTIONS):
    if not p.is_file():raise FileNotFoundError('Add Input dataset chứa '+str(p))
for p in (CACHE,VISUAL_CACHE):
    if not p.is_dir():raise FileNotFoundError('Add Input dataset '+str(p))
def run(args,cwd=None,timeout=900):
    args=list(map(str,args));print('Running:',' '.join(args),flush=True)
    with LOG.open('a',encoding='utf-8') as log:
        log.write('Running: '+' '.join(args)+'\n');log.flush()
        process=subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,errors='replace',bufsize=1,start_new_session=True)
        def drain():
            for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        reader=threading.Thread(target=drain,daemon=True);reader.start();started=time.monotonic()
        try:
            while process.poll() is None:
                if time.monotonic()-started>timeout:raise TimeoutError('Fixed phase timeout; incomplete')
                reader.join(timeout=20)
                if process.poll() is None:print('Worker active;',round(time.monotonic()-started),'s elapsed. See stage/step lines.',flush=True)
            reader.join()
            if process.returncode:raise subprocess.CalledProcessError(process.returncode,args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGTERM)
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()
            reader.join(timeout=5)
print('Bootstrap started; input paths found.',flush=True)


In [ ]:
try:
    run(['git','init',REPO])
    run(['git','remote','add','origin','https://github.com/Supzxjee/Image_Captioning.git'],cwd=REPO)
    run(['git','fetch','--depth','1','origin',SOURCE_COMMIT],cwd=REPO)
    run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO)
    if subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()!=SOURCE_COMMIT:raise ValueError('Wrong source commit')
    for name,h in SOURCE_ASSETS.items():
        if hashlib.sha256((REPO/name).read_bytes()).hexdigest()!=h:raise ValueError('Changed source '+name)
    os.environ['PYTHONPATH']=str(REPO)
    HERE=REPO/'experiments/shared_pair_readout'
    run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocoevalcap==1.2','h5py','numpy','Pillow','nltk==3.9.2','future'])
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
    run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
    run(['java','-version'])
    CHAIR=REPO.parent/'chair'
    run(['git','init',CHAIR]);run(['git','remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],cwd=CHAIR)
    run(['git','fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],cwd=CHAIR)
    run(['git','checkout','--detach','FETCH_HEAD'],cwd=CHAIR)
    if hashlib.sha256((CHAIR/'chair.pkl').read_bytes()).hexdigest()!='73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':raise ValueError('Wrong CHAIR cache')
    CONFIG=REPO.parent/'config.json'
    sys.path.insert(0,str(REPO));sys.path.insert(0,str(HERE))
    from readout_data import POLICY
    cfg=dict(arm=ARM,cache=str(CACHE),checkpoint=str(CHECKPOINT),dataset_json=str(COCO_JSON),
        legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),detections=str(DETECTIONS),
        metadata=str(REPO/'experiments/semantic_grounding_method/data/gain_metadata.json.gz'),
        dimensions=str(HERE/'DEV_DIMENSIONS.json'),chair=str(CHAIR),output=str(OUTPUT),
        policy=POLICY,source_commit=SOURCE_COMMIT,source_assets=SOURCE_ASSETS,
        reported_remaining_gpu_hours=REPORTED_REMAINING_HOURS,
        quota_note='User report25h; not multiplied across accounts; GPU phase cap2device-hours/arm, preprocessing/CPUeval extra.')
    CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')

except BaseException:
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        if LOG.is_file():z.write(LOG,'notebook.log')
        z.writestr('execution_status.json',json.dumps(dict(success=False,arm=ARM,stage='bootstrap',source_commit=SOURCE_COMMIT)))
        z.writestr('bootstrap_error.txt',traceback.format_exc())
    print('BOOTSTRAP FAILED; SEND:',REVIEW,flush=True)
    raise


In [ ]:
success=False
try:
    run([sys.executable,'-u',HERE/'test_readout_data.py'],cwd=REPO)
    run([sys.executable,'-u',HERE/'metric_preflight.py'],cwd=REPO)
    run([sys.executable,'-u',HERE/'run_readout.py','prepare','--config',CONFIG],cwd=REPO,timeout=1800)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
        '--max_restarts=0',HERE/'run_readout.py','worker','--config',CONFIG],cwd=REPO,timeout=3600)
    run([sys.executable,'-u',HERE/'run_readout.py','evaluate','--config',CONFIG],cwd=REPO,timeout=900)
    result=json.loads((OUTPUT/'summary.json').read_text())
    if result['status']!='ENDPOINT_500_EVALUATED_WAIT_MATCHED_I_J':raise RuntimeError('Incomplete endpoint/evaluation')
    success=True
finally:
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        if LOG.is_file():z.write(LOG,'notebook.log')
        z.write(CONFIG,'notebook_config.json')
        for name in SOURCE_ASSETS:
            if name.endswith('.py'):z.write(REPO/name,'source/'+name)
        if OUTPUT.is_dir():
            for p in OUTPUT.iterdir():
                if p.suffix in ('.json','.jsonl','.log'):z.write(p,p.name)
        z.writestr('execution_status.json',json.dumps(dict(success=success,arm=ARM,source_commit=SOURCE_COMMIT,
            assistant_launched_job=False),indent=2))
    print('SEND SMALL REVIEW ZIP:',REVIEW,flush=True)
    if (OUTPUT/'checkpoint.bin').is_file():print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.bin',flush=True)
    if not success:print('INCOMPLETE; send ZIP/log. No result or method improvement accepted.',flush=True)
if success:print('DONE',ARM,'; send BOTH I/J reviewZIPs for paired comparison and semantic panel.',flush=True)


Giữ checkpoint.bin. Gửi `shared_pair_readout_I_review.zip` và `shared_pair_readout_J_review.zip`.
ZIP không chứa checkpoint/HDF5. Endpoint hoàn tất chưa có nghĩa phương pháp tốt hơn hoặc đủ để được nhận bài.
So sánh trên A vừa chạy cùng runtime; nếu khác caption A lịch sử thì báo drift, không thay số lịch sử vào bảng.
Chưa có chấm semantic panel trong notebook. Không tự tạo Dataset, chọn checkpoint, rerank hoặc chạy seed2.
